In [55]:
%use kandy

We are trying to estimate the performance ratio of stdlib functions in Kotlin/Native vs. JVM. For this example, we focus
on `String.substring` method used in this particular scenario. We need to exclude the effect of GC to get the performace
ratio of execution time of the function itself. The following cell describes, how we collect the data. We
end up with a list of execution times.

In [56]:
import java.nio.file.Files
import java.nio.file.Path
import java.nio.file.StandardOpenOption

val kotlincNative = System.getProperty("user.home") + "/.konan/kotlin-native-prebuilt-linux-x86_64-2.2.10/bin/kotlinc-native"
val tempDir = Files.createTempDirectory("knative-perf")

fun runProcess(vararg command: String): String =
    ProcessBuilder(command.toList())
        .directory(tempDir.toFile())
        .start()
        .let { process ->
            process.inputStream.bufferedReader().readText()
                .also {
                    if (process.waitFor() != 0) {
                        process.errorStream.bufferedReader().readText().let(::println)
                        throw Exception("Process failed with code ${process.exitValue()}")
                    }
                }
        }

val source = """
import kotlin.concurrent.Volatile
import kotlin.time.measureTime
import kotlin.time.DurationUnit
import kotlin.time.Duration

@Volatile
private var blackhole: Any? = null
@Volatile
private var string = "abc".repeat(100000)

fun main() {
    repeat(1_000_000) { blackhole = string.substring(1000, 5000) }
    val iterations = 1_000_000
    val times = ArrayList<Duration>(iterations)
    repeat(iterations) {
        measureTime {
            repeat(100) {
                blackhole = string.substring(1000, 5000)
            }
        }.let { times.add(it) }
    }
    times.forEach { println(it.toDouble(DurationUnit.SECONDS)) }
}
"""

val sourceFile = tempDir.resolve("benchmark.kt")
val outputBinary = tempDir.resolve("benchmark.kexe")

Files.writeString(
    sourceFile,
    source,
    StandardOpenOption.CREATE,
)

runProcess(
        kotlincNative,
        sourceFile.toString(),
        "-o", outputBinary.toString()
)

val times = runProcess(outputBinary.toString())
    .split("\n")
    .mapNotNull { it.takeIf { it.isNotEmpty() }?.toDouble() }
    .toList()

For the exclusion of GC, we assume that the execution time distribution is bimodal (i .e. there are two peaks). The
first peak is formed by the executions that didn't trigger GC. The second peak is formed by the executions that
triggered GC. By finding a threshold for the execution time that lies between the two peaks, we can separate the
executions into those that ran GC and those which didn't.

In [57]:
val threshold = 0.00005

In [58]:
plot {
    histogram(times, binsOption = BinsOption.byNumber(100))
    vLine {
        xIntercept.constant(threshold)
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="A4Dosd"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("A4Dosd");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"x",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count"
},
"stat":"identity",
"data":{
"x":[8.177752E-5,2.4533256E-4,4.088876E-4,5.7244264E-4,7.3599768E-4,8.9955272E-4,0.00106310776,0.0012266628,0.00139021784,0.0015537728799999999,0.0017173279199999998,0.00188088296,0.002044438,0.00220799304,0.00237154808,0.00253510312,0.00269865816,0.0028622132,0.0030257682399999998,0.0031893232799999997,0.0033528783199999997,0.0035164333599999997,0.0036799883999999996,0.0038435434399999996,0.00400709848,0.0041706535199999995,0.00433420856,0.0044977635999999994,0.00466131864,0.004824873679999999,0.00498842872,0.005151983759999999,0.0053155388,0.005479093839999999,0.00564264888,0.005806203919999999,0.00596975896,0.006133313999999999,0.00629686904,0.006460424079999999,0.00662397912,0.006787534159999999,0.0069510892,0.007114644239999999,0.00727819928,0.007441754319999999,0.00760530936,0.0077688644,0.00793241944,0.00809597448,0.00825952952,0.008423084559999999,0.0085866396,0.00875019464,0.00891374968,0.009077304719999998,0.00924085976,0.0094044148,0.00956796984,0.009731524879999998,0.009895079919999999,0.01005863496,0.010222189999999999,0.010385745039999998,0.01054930008,0.01071285512,0.010876410159999999,0.0110399652,0.01120352024,0.01136707528,0.011530630319999999,0.01169418536,0.0118577404,0.01202129544,0.012184850479999999,0.01234840552,0.01251196056,0.0126755156,0.012839070639999998,0.01300262568,0.01316618072,0.01332973576,0.013493290799999998,0.01365684584,0.01382040088,0.013983955919999999,0.014147510959999998,0.014311065999999999,0.01447462104,0.014638176079999999,0.014801731119999998,0.014965286159999999,0.0151288412,0.015292396239999999,0.015455951279999998,0.01561950632,0.01578306136,0.0159466164,0.01611017144,0.01627372648],
"count":[930320.0,3155.0,2652.0,3135.0,3415.0,3291.0,3106.0,2837.0,2658.0,2794.0,3019.0,3019.0,2583.0,3029.0,2222.0,1473.0,1463.0,1263.0,1360.0,1364.0,1185.0,1139.0,1017.0,942.0,819.0,1237.0,935.0,863.0,1059.0,939.0,919.0,925.0,1033.0,923.0,1042.0,648.0,301.0,509.0,633.0,786.0,615.0,512.0,517.0,282.0,411.0,395.0,353.0,300.0,257.0,191.0,54.0,20.0,11.0,16.0,7.0,5.0,4.0,6.0,6.0,3.0,2.0,5.0,3.0,3.0,1.0,0.0,1.0,2.0,0.0,0.0,3.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"int",
"column":"count"
}]
}
},{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"xintercept":5.0E-5,
"geom":"vline",
"data":{
}
}],
"spec_id":"20"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || 

We then estimate the total time spent in GC by the following formula.

In [49]:
val (noGC, GC) = times.partition { it < threshold }
val avgNoGC = noGC.average()
GC.sumOf { it - avgNoGC } / times.sum()

0.8721033399174649

For measurements that run at least 100 000 iterations and don't print the times throught the execution (but all at once at the end), we estimate that GC is responsible for over 80% of the execution. That is insane. Therefore, we question the results.

* Can GC in K/N be responsible for such a high percentage of execution time? After all, this is very memory heavy benchmark.
* Is there an error in the benchmark setup, collection of the data or the processing?
* Is any one of our assumptions incorrect?
* Is this method for estimating GC time sound?